# 39 — Regulatory Change Tracker

A delta-aware compliance pipeline that diffs a new regulatory update against an existing obligation register, identifies only the net-new obligations not already tracked, then cross-references each one against a contract corpus to score exposure severity and draft concrete remediation actions. After each run the pipeline writes a new versioned `ComplianceState` object — preserving immutability across update cycles so the full audit trail is retained.

## Framework Comparison

The core pattern demonstrated here is **delta-aware stateful extraction**: a three-stage pipeline where each stage is a typed LLM call, obligations are diffed (not blindly enumerated), exposure checks run in parallel, and state is versioned rather than mutated.

| Dimension | This example (OpenAI SDK) | LangGraph equivalent | LangChain equivalent | CrewAI equivalent |
|-----------|--------------------------|---------------------|---------------------|------------------|
| **State object** | `ComplianceState` (Pydantic, versioned immutably per run) | `StateAnnotation` typed dict passed through graph nodes | `ConversationBufferMemory` or a custom `BaseMemory` store | Shared `crew.context` dict mutated across tasks |
| **Diff / retrieval step** | `_extract_obligations()` — LLM reads update + register and returns only net-new items | A graph node with a retrieval edge that filters existing state before calling the LLM | RAG retrieval step (`VectorStoreRetriever`) injecting existing obligations as context before the extraction chain | A `Task` with context from a prior task that lists current obligations so the agent can exclude duplicates |
| **Parallel exposure checks** | `ThreadPoolExecutor` — one LLM call per new obligation, results gathered with `as_completed` | Parallel fan-out using `Send` edges to sibling nodes, each handling one obligation | `RunnableParallel` (LCEL) mapping one chain per obligation and collecting results | Multiple `Agent` instances assigned in parallel via `crew.kickoff()` with `process=Process.hierarchical` |

In [ ]:
!pip install openai pydantic python-dotenv

In [ ]:
import os

# Replace the value below with your actual OpenAI API key before running
os.environ["OPENAI_API_KEY"] = "sk-..."

In [ ]:
# ruff: noqa: E402
"""Pydantic models for the regulatory change tracker."""

from typing import Literal

from pydantic import BaseModel, Field


class Obligation(BaseModel):
    """A single compliance obligation in the existing register."""

    id: str = Field(description="Unique identifier for the obligation (e.g. OBL-001).")
    text: str = Field(description="Full text of the obligation as written in the source regulation.")
    source_article: str = Field(
        description="Article or section reference in the regulation (e.g. 'Article 17(1)')."
    )
    category: str = Field(
        description="Obligation category (e.g. 'data-retention', 'consent', 'disclosure')."
    )
    effective_date: str = Field(
        description="ISO date from which the obligation is effective (e.g. '2024-01-01')."
    )
    status: Literal["active", "pending", "superseded"] = Field(
        description="Lifecycle status of the obligation."
    )


class ObligationRegister(BaseModel):
    """The existing set of tracked compliance obligations before the new update."""

    jurisdiction: str = Field(
        description="Jurisdiction covered by this register (e.g. 'EU', 'UK', 'US-CA')."
    )
    obligations: list[Obligation] = Field(
        description="All obligations currently tracked in the register."
    )


class RegulatoryUpdate(BaseModel):
    """A new regulatory update to be ingested and diffed against the register."""

    update_id: str = Field(
        description="Unique identifier for this regulatory update (e.g. 'GDPR-AMD-2024-03')."
    )
    title: str = Field(description="Human-readable title of the regulatory update.")
    effective_date: str = Field(
        description="ISO date from which the update takes effect (e.g. '2025-01-01')."
    )
    jurisdiction: str = Field(
        description="Jurisdiction the update applies to (e.g. 'EU', 'UK', 'US-CA')."
    )
    raw_text: str = Field(
        description="Full text of the regulatory update as received from the official source."
    )
    summary: str = Field(
        description="A 1-3 sentence plain-English summary of what has changed in this update."
    )


class NewObligation(BaseModel):
    """A net-new obligation extracted from the regulatory update — not yet in the register."""

    text: str = Field(
        description="Full text of the new obligation as extracted from the update."
    )
    source_article: str = Field(
        description="Article or section reference in the update that introduces this obligation."
    )
    category: str = Field(
        description="Obligation category (e.g. 'data-retention', 'breach-notification', 'audit')."
    )
    effective_date: str = Field(
        description="ISO date from which this obligation is effective."
    )


class ContractExposure(BaseModel):
    """Exposure assessment for a single contract against one new obligation."""

    contract_name: str = Field(
        description="Human-readable name or identifier of the assessed contract."
    )
    exposed_clauses: list[str] = Field(
        description=(
            "List of clause references or short excerpts in the contract that conflict with "
            "or are inadequate for the new obligation. Empty list if no exposure found."
        )
    )
    impact: Literal["none", "moderate", "severe"] = Field(
        description=(
            "Severity of exposure: 'none' means the contract already satisfies the obligation, "
            "'moderate' means minor gaps requiring amendment, "
            "'severe' means fundamental conflict or missing provisions."
        )
    )
    remediation_action: str = Field(
        description=(
            "Specific action required to bring the contract into compliance. "
            "Use 'No action required' when impact is none."
        )
    )


class ObligationImpact(BaseModel):
    """Full impact assessment for one net-new obligation across all contracts."""

    obligation: NewObligation = Field(
        description="The net-new obligation being assessed."
    )
    contract_exposures: list[ContractExposure] = Field(
        description="Exposure assessments for each contract in the corpus."
    )
    overall_impact: Literal["none", "moderate", "severe"] = Field(
        description=(
            "Rolled-up severity across all contracts: the highest severity level "
            "found in any single contract exposure."
        )
    )
    action_item: str = Field(
        description=(
            "A single, prioritised remediation action item to address the most "
            "severe exposure. Should be specific enough to assign to a person."
        )
    )


class ComplianceState(BaseModel):
    """Versioned compliance tracker object — updated after each regulatory change cycle."""

    version: int = Field(
        description="Monotonically increasing version number incremented on every update."
    )
    last_updated: str = Field(
        description="ISO date of the most recent update to this state object (e.g. '2025-01-15')."
    )
    jurisdiction: str = Field(
        description="Jurisdiction covered by this compliance state."
    )
    obligations: list[Obligation] = Field(
        description="Complete list of all active and pending obligations after this update cycle."
    )
    pending_actions: list[str] = Field(
        description="Ordered list of outstanding remediation action items across all obligations."
    )
    last_update_summary: str = Field(
        description="Plain-English summary of what changed in the most recent update cycle."
    )


class ChangeTrackerResult(BaseModel):
    """Final output of one regulatory change tracking run."""

    update_id: str = Field(
        description="Identifier of the regulatory update that was processed."
    )
    net_new_count: int = Field(
        description="Number of net-new obligations identified that were not in the prior register."
    )
    impact_assessments: list[ObligationImpact] = Field(
        description="Full impact assessment for each net-new obligation."
    )
    updated_state: ComplianceState = Field(
        description="The versioned compliance state object after incorporating this update."
    )
    summary: str = Field(
        description=(
            "2-4 sentence executive summary: how many new obligations were found, "
            "highest severity level encountered, and top priority action."
        )
    )

In [ ]:
"""System prompt constants for the regulatory change tracker."""

EXTRACTION_SYSTEM = """You are a legal compliance analyst specialising in regulatory change management.

Your task is to diff a new regulatory update against an existing obligation register and identify
ONLY the net-new obligations that are NOT already captured in the register.

Input (JSON):
- "update": the new regulatory update with fields: update_id, title, effective_date, jurisdiction,
  raw_text, summary
- "existing_obligations": array of obligations already in the register, each with: id, text,
  source_article, category, effective_date, status

Rules:
- Read the update's raw_text carefully and extract every distinct compliance obligation.
- Diff each extracted obligation against the existing_obligations list.
  An obligation is "existing" if its substance is substantially covered by any active or pending
  entry in the register — minor wording differences do not make it new.
- Return ONLY obligations that are genuinely net-new (not already in the register).
- For each net-new obligation, populate all fields: text, source_article, category, effective_date.
- source_article must cite the specific article, paragraph, or section in the update raw_text.
- category must be a short kebab-case label (e.g. data-retention, breach-notification, audit-trail,
  consent-management, cross-border-transfer, record-keeping, risk-assessment).
- effective_date must be an ISO date string (YYYY-MM-DD).
- If the update contains no net-new obligations, return an empty array.
- Return ONLY valid JSON: an array of NewObligation objects — no prose, no markdown fences."""

EXPOSURE_SYSTEM = """You are a contract review specialist assessing compliance exposure.

Your task is to evaluate a single new regulatory obligation against a set of contract excerpts
and determine which contracts are exposed, how severely, and what remediation is needed.

Input (JSON):
- "obligation": a new regulatory obligation with fields: text, source_article, category,
  effective_date
- "contracts": array of contract excerpts, each with: contract_name, excerpt

Rules:
- For each contract, assess whether the contract's existing language satisfies the new obligation.
- Score impact as:
    "none"     — the contract already contains adequate provisions for this obligation
    "moderate" — the contract has relevant provisions but they are incomplete or require amendment
    "severe"   — the contract has no provisions for this obligation or directly conflicts with it
- exposed_clauses: list the specific clause numbers or short excerpt snippets that are deficient.
  Use an empty list when impact is "none".
- remediation_action: write a concrete, assignable action (e.g. "Add a data retention clause
  capping storage at 24 months per Article 5(1)(e) to the DataSync SaaS Agreement").
  Use "No action required" only when impact is "none".
- Be conservative: when in doubt between moderate and severe, prefer severe.
- Return ONLY valid JSON: an array of ContractExposure objects — no prose, no markdown fences."""

STATE_UPDATE_SYSTEM = """You are a compliance programme manager maintaining a versioned obligation register.

Your task is to update an existing ComplianceState object to reflect newly assessed obligations
from a regulatory update cycle.

Input (JSON):
- "existing_state": the current ComplianceState with fields: version, last_updated, jurisdiction,
  obligations, pending_actions, last_update_summary
- "assessments": array of ObligationImpact objects, each containing: obligation (NewObligation),
  contract_exposures (list of ContractExposure), overall_impact, action_item
- "update_id": identifier of the regulatory update just processed
- "today": today's ISO date (YYYY-MM-DD)

Rules:
- version: increment by 1.
- last_updated: set to the value of "today".
- jurisdiction: carry forward unchanged.
- obligations: append each net-new obligation as a new Obligation entry. Assign a new id using
  the pattern "OBL-NNN" where NNN is the next sequential number after the highest existing id.
  Set status to "pending" for all newly added obligations.
  Carry forward all existing obligations unchanged.
- pending_actions: append the action_item from every assessment where overall_impact is not "none".
  Preserve any existing pending_actions. Deduplicate if the same action already appears.
- last_update_summary: write 2-3 sentences describing what changed: how many obligations were
  added, the highest impact level found, and the jurisdiction.
- Return ONLY valid JSON matching the ComplianceState schema — no prose, no markdown fences."""

In [ ]:
# ruff: noqa: E402
"""Regulatory change tracker workflow — delta-aware stateful extraction."""

import json
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import date

from openai import OpenAI

_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
_MODEL = "gpt-4o-mini"


def _extract_obligations(
    update: RegulatoryUpdate,
    register: ObligationRegister,
) -> list[NewObligation]:
    """Stage 1: diff the update against the register; return only net-new obligations."""
    user_message = json.dumps(
        {
            "update": update.model_dump(),
            "existing_obligations": [o.model_dump() for o in register.obligations],
        }
    )
    array_schema = {
        "type": "object",
        "properties": {
            "obligations": {
                "type": "array",
                "items": NewObligation.model_json_schema(),
            }
        },
        "required": ["obligations"],
        "additionalProperties": False,
    }
    resp = _client.chat.completions.create(
        model=_MODEL,
        messages=[
            {"role": "system", "content": EXTRACTION_SYSTEM},
            {"role": "user", "content": user_message},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "NewObligationList",
                "strict": True,
                "schema": array_schema,
            },
        },
    )
    data = json.loads(resp.choices[0].message.content)
    return [NewObligation.model_validate(o) for o in data["obligations"]]


def _assess_exposure(
    obligation: NewObligation,
    contracts: list[dict],
) -> ObligationImpact:
    """Stage 2: assess one net-new obligation against the full contract corpus."""
    user_message = json.dumps(
        {
            "obligation": obligation.model_dump(),
            "contracts": contracts,
        }
    )
    array_schema = {
        "type": "object",
        "properties": {
            "exposures": {
                "type": "array",
                "items": ContractExposure.model_json_schema(),
            }
        },
        "required": ["exposures"],
        "additionalProperties": False,
    }
    resp = _client.chat.completions.create(
        model=_MODEL,
        messages=[
            {"role": "system", "content": EXPOSURE_SYSTEM},
            {"role": "user", "content": user_message},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "ContractExposureList",
                "strict": True,
                "schema": array_schema,
            },
        },
    )
    data = json.loads(resp.choices[0].message.content)
    exposures = [ContractExposure.model_validate(e) for e in data["exposures"]]

    _severity_rank = {"none": 0, "moderate": 1, "severe": 2}
    overall = (
        max(exposures, key=lambda e: _severity_rank[e.impact]).impact
        if exposures
        else "none"
    )
    severe_or_moderate = [e for e in exposures if e.impact != "none"]
    action = (
        severe_or_moderate[0].remediation_action
        if severe_or_moderate
        else "No action required"
    )
    return ObligationImpact(
        obligation=obligation,
        contract_exposures=exposures,
        overall_impact=overall,
        action_item=action,
    )


def _update_state(
    state: ComplianceState,
    assessments: list[ObligationImpact],
    update: RegulatoryUpdate,
) -> ComplianceState:
    """Stage 3: produce a new versioned ComplianceState incorporating the assessed obligations."""
    user_message = json.dumps(
        {
            "existing_state": state.model_dump(),
            "assessments": [a.model_dump() for a in assessments],
            "update_id": update.update_id,
            "today": date.today().isoformat(),
        }
    )
    resp = _client.chat.completions.create(
        model=_MODEL,
        messages=[
            {"role": "system", "content": STATE_UPDATE_SYSTEM},
            {"role": "user", "content": user_message},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "ComplianceState",
                "strict": True,
                "schema": ComplianceState.model_json_schema(),
            },
        },
    )
    return ComplianceState.model_validate_json(resp.choices[0].message.content)


def run(
    update: RegulatoryUpdate,
    register: ObligationRegister,
    contracts: list[dict],
    state: ComplianceState,
) -> ChangeTrackerResult:
    """Run the full regulatory change tracking pipeline.

    Args:
        update: The incoming regulatory update to process.
        register: The existing obligation register to diff against.
        contracts: List of contract excerpts, each a dict with keys
                   ``contract_name`` and ``excerpt``.
        state: The current versioned ComplianceState to update.

    Returns:
        A ChangeTrackerResult containing the net-new count, per-obligation
        impact assessments, updated ComplianceState, and an executive summary.
    """
    # Stage 1 — extract net-new obligations via diff
    new_obligations = _extract_obligations(update, register)

    # Stage 2 — assess exposure for each new obligation in parallel
    assessments: list[ObligationImpact] = []
    if new_obligations:
        with ThreadPoolExecutor(max_workers=min(len(new_obligations), 5)) as executor:
            futures = {
                executor.submit(_assess_exposure, obl, contracts): obl
                for obl in new_obligations
            }
            for future in as_completed(futures):
                assessments.append(future.result())

    # Stage 3 — write versioned state update
    updated_state = _update_state(state, assessments, update)

    _severity_rank = {"none": 0, "moderate": 1, "severe": 2}
    top_impact = (
        max(assessments, key=lambda a: _severity_rank[a.overall_impact]).overall_impact
        if assessments
        else "none"
    )
    top_action = (
        next(
            (a.action_item for a in assessments if a.overall_impact == top_impact),
            "No action required",
        )
        if top_impact != "none"
        else "No action required"
    )
    summary = (
        f"Processed {update.update_id}: {len(new_obligations)} net-new obligation(s) identified "
        f"for {update.jurisdiction}. "
        f"Highest exposure level: {top_impact}. "
        f"Top priority action: {top_action}"
    )

    return ChangeTrackerResult(
        update_id=update.update_id,
        net_new_count=len(new_obligations),
        impact_assessments=assessments,
        updated_state=updated_state,
        summary=summary,
    )

In [ ]:
# ---------------------------------------------------------------------------
# Scenario A — GDPR Amendment: new data retention cap obligation
# Existing register already tracks consent + right-to-erasure.
# The amendment adds a net-new 24-month retention cap; one contract is exposed.
# ---------------------------------------------------------------------------

SCENARIO_A_UPDATE = RegulatoryUpdate(
    update_id="GDPR-AMD-2025-01",
    title="GDPR Amendment — Article 5 Data Retention Cap (24 months)",
    effective_date="2025-07-01",
    jurisdiction="EU",
    raw_text=(
        "Amendment to Regulation (EU) 2016/679 (GDPR) — Article 5 Data Quality Principles\n\n"
        "Article 5(1)(e) is amended as follows:\n\n"
        "Personal data shall be kept in a form which permits identification of data subjects "
        "for no longer than is necessary for the purposes for which the personal data are "
        "processed ('storage limitation'). With effect from 1 July 2025, controllers must "
        "implement a maximum retention period of 24 months for all personal data processed "
        "under a legitimate interest basis, unless a longer period is mandated by Union or "
        "Member State law. Controllers must document the retention schedule and make it "
        "available to supervisory authorities on request.\n\n"
        "Article 13(2)(a) is amended to require that privacy notices explicitly state "
        "the 24-month maximum retention period where applicable.\n\n"
        "No changes are made to Articles 6 (lawful basis), 7 (consent conditions), or "
        "17 (right to erasure) by this amendment."
    ),
    summary=(
        "Introduces a hard 24-month maximum retention period for personal data processed "
        "under legitimate interest. Controllers must document retention schedules and update "
        "privacy notices to reflect the cap."
    ),
)

SCENARIO_A_REGISTER = ObligationRegister(
    jurisdiction="EU",
    obligations=[
        Obligation(
            id="OBL-001",
            text=(
                "Controllers must obtain freely given, specific, informed and unambiguous "
                "consent before processing personal data where consent is the lawful basis."
            ),
            source_article="Article 7(1) GDPR",
            category="consent-management",
            effective_date="2018-05-25",
            status="active",
        ),
        Obligation(
            id="OBL-002",
            text=(
                "Controllers must erase personal data without undue delay where the data "
                "subject withdraws consent or objects to processing, and no other lawful "
                "basis applies."
            ),
            source_article="Article 17(1) GDPR",
            category="data-erasure",
            effective_date="2018-05-25",
            status="active",
        ),
    ],
)

SCENARIO_A_CONTRACTS = [
    {
        "contract_name": "DataSync SaaS Agreement v3.2",
        "excerpt": (
            "Clause 8.1 — Data Retention: The Processor shall retain Customer Personal Data "
            "for the duration of the contract term and for a period of 36 months thereafter, "
            "unless otherwise instructed by the Controller in writing. "
            "Clause 8.2 — Upon termination, the Processor shall delete or return all "
            "Customer Personal Data within 30 days of written request."
        ),
    },
    {
        "contract_name": "AnalyticsPro Licence Agreement v1.0",
        "excerpt": (
            "Section 5 — Privacy: The parties agree to comply with applicable data protection "
            "legislation. The Licensee is responsible for obtaining appropriate consents from "
            "end users prior to processing their personal data through the AnalyticsPro platform. "
            "Data processed under this licence shall be subject to the Licensee's privacy policy."
        ),
    },
]

SCENARIO_A_STATE = ComplianceState(
    version=3,
    last_updated="2024-11-15",
    jurisdiction="EU",
    obligations=SCENARIO_A_REGISTER.obligations,
    pending_actions=[
        "Review consent capture flows against Article 7(1) requirements before Q1 audit.",
    ],
    last_update_summary=(
        "Processed GDPR-GUIDE-2024-11: no net-new obligations identified. "
        "Guidance update on right-to-erasure timelines — existing OBL-002 remains adequate."
    ),
)

print("Running Scenario A: GDPR Amendment — Data Retention Cap")
result_a = run(
    SCENARIO_A_UPDATE, SCENARIO_A_REGISTER, SCENARIO_A_CONTRACTS, SCENARIO_A_STATE
)
print(result_a.model_dump_json(indent=2))

In [ ]:
# ---------------------------------------------------------------------------
# Scenario B — MiFID II Amendment: new best-execution reporting obligation
# Existing register already captures transaction reporting + client categorisation.
# The amendment adds a net-new quarterly best-execution report; both contracts
# have generic compliance clauses that mostly satisfy the new requirement.
# ---------------------------------------------------------------------------

SCENARIO_B_UPDATE = RegulatoryUpdate(
    update_id="MIFID-AMD-2025-03",
    title="MiFID II Delegated Regulation — Quarterly Best-Execution Report (RTS 28 Revision)",
    effective_date="2025-10-01",
    jurisdiction="EU-MiFID",
    raw_text=(
        "Commission Delegated Regulation amending RTS 28 under MiFID II\n\n"
        "Article 3 — Quarterly Best-Execution Report\n"
        "Investment firms shall publish, on a quarterly basis and no later than 15 business "
        "days after the end of each calendar quarter, a report summarising the top five "
        "execution venues used for each class of financial instrument, together with "
        "information on the quality of execution obtained. The report must be published "
        "in machine-readable format on the firm's public website and submitted to the "
        "competent authority via the designated reporting portal.\n\n"
        "Article 5 — Transaction Reporting (unchanged)\n"
        "No changes are made to Article 26 MiFID II transaction reporting obligations. "
        "Existing transaction reporting schedules and formats remain in effect.\n\n"
        "Article 7 — Client Categorisation (unchanged)\n"
        "Client categorisation requirements under Articles 4(1)(10-12) MiFID II are "
        "not affected by this amendment."
    ),
    summary=(
        "Revises RTS 28 to require a quarterly best-execution report published within "
        "15 business days of each quarter end, in machine-readable format. "
        "Transaction reporting and client categorisation obligations are unchanged."
    ),
)

SCENARIO_B_REGISTER = ObligationRegister(
    jurisdiction="EU-MiFID",
    obligations=[
        Obligation(
            id="OBL-001",
            text=(
                "Investment firms must submit transaction reports to the competent authority "
                "by the close of business on the working day following execution of a "
                "reportable transaction."
            ),
            source_article="Article 26 MiFID II",
            category="transaction-reporting",
            effective_date="2018-01-03",
            status="active",
        ),
        Obligation(
            id="OBL-002",
            text=(
                "Investment firms must categorise each client as a retail client, "
                "professional client, or eligible counterparty and maintain records of "
                "the categorisation and the basis on which it was made."
            ),
            source_article="Article 4(1)(10-12) MiFID II",
            category="client-categorisation",
            effective_date="2018-01-03",
            status="active",
        ),
    ],
)

SCENARIO_B_CONTRACTS = [
    {
        "contract_name": "Prime Brokerage Services Agreement v2.1",
        "excerpt": (
            "Schedule 3 — Regulatory Compliance: Each party agrees to maintain policies "
            "and procedures reasonably designed to comply with applicable MiFID II requirements "
            "as amended from time to time. The Broker shall publish best-execution policies "
            "annually and make them available to the Client on request. "
            "Clause 12.4: The Broker maintains an order execution policy reviewed at least "
            "annually and updated to reflect material changes in regulatory requirements."
        ),
    },
    {
        "contract_name": "Algorithmic Trading Platform Agreement v1.5",
        "excerpt": (
            "Section 9 — Reporting Obligations: The Platform Provider will assist the Client "
            "in meeting its regulatory reporting obligations, including transaction reporting "
            "under Article 26 MiFID II. The Provider maintains connectivity to approved "
            "reporting mechanisms (ARMs) and will submit reports on the Client's behalf. "
            "Section 9.3: Best-execution data is available via the Provider's API in "
            "CSV and JSON formats for Client's own reporting requirements."
        ),
    },
]

SCENARIO_B_STATE = ComplianceState(
    version=5,
    last_updated="2025-01-20",
    jurisdiction="EU-MiFID",
    obligations=SCENARIO_B_REGISTER.obligations,
    pending_actions=[
        "Schedule annual review of order execution policy before December 2025.",
    ],
    last_update_summary=(
        "Processed MIFID-GUIDE-2025-01: no net-new obligations from ESMA Q4 guidance. "
        "Existing transaction reporting and categorisation obligations confirmed adequate."
    ),
)

print("Running Scenario B: MiFID II Amendment — Best-Execution Reporting")
result_b = run(
    SCENARIO_B_UPDATE, SCENARIO_B_REGISTER, SCENARIO_B_CONTRACTS, SCENARIO_B_STATE
)
print(result_b.model_dump_json(indent=2))

## Starter Exercise

Extend the tracker to handle **conflicting** obligations — where a new update REMOVES or RELAXES a prior obligation. How would you modify the schema and extraction prompt to surface both additions and removals?

Currently `NewObligation` only models obligations being *added*. But regulators also issue amendments that:
- **Remove** an obligation entirely (e.g. a data minimisation rule is repealed)
- **Relax** an existing obligation (e.g. a 30-day breach notification window is extended to 72 hours)

These are silent risks: without surfacing them, the register retains stale obligations that no longer apply, and teams spend effort on compliance work that is no longer required.

Think through:
1. What field would you add to `NewObligation` to distinguish additions, removals, and relaxations?
2. How would you update `EXTRACTION_SYSTEM` to instruct the LLM to detect and classify all three types?
3. How would `_update_state` need to change to handle obligations that should be marked `superseded` rather than appended?

In [ ]:
# ---------------------------------------------------------------------------
# Answer key: surfacing obligation removals and relaxations
# ---------------------------------------------------------------------------

# Step 1 — extend NewObligation with an obligation_type discriminator
from typing import Literal
from pydantic import BaseModel, Field


class NewObligationV2(BaseModel):
    """Extended obligation model that captures additions, removals, and relaxations."""

    text: str = Field(
        description="Full text of the new or modified obligation as extracted from the update."
    )
    source_article: str = Field(
        description="Article or section reference in the update."
    )
    category: str = Field(
        description="Obligation category in kebab-case (e.g. 'data-retention')."
    )
    effective_date: str = Field(
        description="ISO date from which this change is effective."
    )
    # NEW FIELD — the key addition for conflict-aware tracking
    obligation_type: Literal["add", "remove", "relax"] = Field(
        description=(
            "Type of change introduced by this obligation: "
            "'add' means a brand-new obligation not previously in the register; "
            "'remove' means an existing obligation is fully repealed by this update; "
            "'relax' means an existing obligation is loosened (e.g. longer deadline, "
            "narrower scope, lower threshold) but not fully repealed."
        )
    )
    # Optional: for remove/relax, reference the existing obligation being affected
    supersedes_obligation_id: str | None = Field(
        default=None,
        description=(
            "For 'remove' and 'relax' types: the id of the existing Obligation in the register "
            "that this change supersedes or modifies. Null for 'add' types."
        )
    )


# Step 2 — updated extraction prompt excerpt showing how to classify all three types
EXTRACTION_SYSTEM_V2_ADDITION = """
For each obligation change found in the update raw_text, set obligation_type as follows:
  - "add"    : the obligation is genuinely new — it does not appear in the existing register.
  - "remove" : the update explicitly repeals, deletes, or removes an obligation that IS
               currently in the register. Set supersedes_obligation_id to the id of the
               register entry being repealed.
  - "relax"  : the update modifies an existing obligation to make it less stringent
               (e.g. extends a deadline, narrows scope, reduces a threshold). Set
               supersedes_obligation_id to the id of the register entry being relaxed.
Return ALL three types. Do not skip removals or relaxations — they are as important as additions.
"""

# Step 3 — how _update_state would change for remove/relax obligations
# (pseudocode — the real implementation would pass this logic to the STATE_UPDATE_SYSTEM prompt)
def _apply_obligation_delta(state: ComplianceState, obligation: NewObligationV2) -> ComplianceState:
    """Pseudocode showing how each obligation_type modifies state."""
    obligations = list(state.obligations)  # work on a copy — preserve immutability

    if obligation.obligation_type == "add":
        # Append as a new pending obligation (existing behaviour)
        pass  # handled by STATE_UPDATE_SYSTEM prompt

    elif obligation.obligation_type == "remove":
        # Mark the superseded obligation as 'superseded' rather than active/pending
        obligations = [
            obl.model_copy(update={"status": "superseded"})
            if obl.id == obligation.supersedes_obligation_id
            else obl
            for obl in obligations
        ]

    elif obligation.obligation_type == "relax":
        # Supersede the old entry and append the relaxed version as the new canonical obligation
        obligations = [
            obl.model_copy(update={"status": "superseded"})
            if obl.id == obligation.supersedes_obligation_id
            else obl
            for obl in obligations
        ]
        # The relaxed obligation would then be appended as a new 'pending' entry
        # (with the updated text and effective_date from the regulatory update)

    return state.model_copy(update={"obligations": obligations})


print("NewObligationV2 schema:")
print(NewObligationV2.model_json_schema())